# 02 · Close preprocessing (amendment 1.2)

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

This notebook finishes the preprocessing stage before any model is trained. It
1. rebuilds UNSW-NB15 and NSL-KDD with the current (1.1) rules, because their outputs came from the 1.0 code;
2. measures exact and near-duplicate overlap between CICIDS2017 train and the external targets again;
3. builds the near-duplicate records for sensitivity S7;
4. verifies every output: built under the current rules, not a pilot, and every file's SHA-256 matches its report.

Run every cell **from top to bottom**, one at a time, waiting for each to finish. Nothing to add or edit.
**Before you start:** *Runtime → Change runtime type → High-RAM*.

| Step | What it does | Time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests (expected `33 passed`) | 2 min |
| 2 | UNSW-NB15 | 10–15 min |
| 3 | NSL-KDD | 2 min |
| 4 | cross-dataset overlap, exact and near-duplicate | 45–90 min |
| 5 | near-duplicate records for CICIDS2017, CICIoMT2024, ToN_IoT, CIC-IoT2023, NSL-KDD | 30–60 min |
| 6 | verify all outputs (expected last line `all outputs verified`) | 20–40 min |
| 7 | summary tables | 1 min |

Each step saves its results to Drive and pushes them to GitHub before the next one starts. **If Colab
disconnects:** run step 1 again, then continue from the step that did not finish.

**Where results go:** `results/data_audit/` (reports, `near_duplicates.json`, `cross_dataset_overlap.json`,
`verification.md`) and `docs/data_cards/`, in Drive at `MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/` and on
GitHub; data files in `MyDrive/PhD-AI-Cyber/data/processed/` and `data/splits/` (Drive only).


## Step 1 · Connect and update (expected end: `in sync with GitHub at …` and `33 passed`)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, pathlib, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

import datetime as _dt, os, pathlib, shutil, subprocess

RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')

def git(*args, check=False):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(TOKEN or '\0', '***').strip()
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def sync(message=None):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    """
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    git('rebase', '--abort'); git('merge', '--abort')      # clear any half-finished earlier pull
    for attempt in range(3):
        git('fetch', REMOTE, 'main', check=True)
        base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
        changed = set(git('diff', '--name-only', base)[1].split())
        untracked = set(git('ls-files', '--others', '--exclude-standard')[1].split())
        # untracked non-record files survive the reset unless GitHub has a file at the same path
        untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
        changed |= untracked
        keep = sorted(f for f in changed if f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        other = sorted(f for f in changed if not f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        stamp = _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        hold = pathlib.Path('/content/sync_hold') / stamp if pathlib.Path('/content').exists() else pathlib.Path(REPO_DIR).parent / f'.sync_hold_{stamp}'
        for f in keep:
            (hold / f).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
        if other:
            bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
            for f in other:
                (bak / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
            print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
        git('reset', '--hard', 'FETCH_HEAD', check=True)
        for f in keep:
            dest = pathlib.Path(REPO_DIR, f)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(hold / f, dest)
        if keep:
            git('add', '--', *keep, check=True)
            if git('diff', '--cached', '--quiet')[0] != 0:
                git('commit', '-m', message or 'Save results produced in Colab', check=True)
        code, out = git('push', REMOTE, 'HEAD:main')
        if code == 0:
            head = git('log', '--oneline', '-1')[1]
            print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
            return
        print('GitHub moved while saving; trying again')
    raise SystemExit('could not save to GitHub after 3 tries: copy the messages above into the project thread')


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'src/data/pipeline.py').exists()
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests

def push(message):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records."""
    sync(message)
    print('saved to GitHub:', message)

def run(*args, message):
    """Run one preprocessing command (its output shows below); push the results only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/preprocess.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')


## Step 2 · UNSW-NB15 (rebuilt under the 1.1 rules)

In [ ]:
run('--dataset', 'unsw_nb15', message='Preprocess UNSW-NB15 (protocol 1.1)')

## Step 3 · NSL-KDD (rebuilt under the 1.1 rules)

In [ ]:
run('--dataset', 'nsl_kdd', message='Preprocess NSL-KDD (protocol 1.1)')

## Step 4 · Cross-dataset overlap, exact and near-duplicate (CICIDS2017 train vs CSE-CIC-IDS2018 and UNSW-NB15)

In [ ]:
run('--cross-overlap', message='Cross-dataset overlap, exact and near-duplicate (amendment 1.2)')

## Step 5 · Near-duplicate records for sensitivity S7

Stops with an error if the recomputed near-duplicates do not match the preprocessing reports exactly.

In [ ]:
run('--near-duplicates', message='Near-duplicate records for sensitivity S7 (amendment 1.2)')

## Step 6 · Verify every output (rules, pilot flag, SHA-256 of every file)

In [ ]:
run('--verify', message='Verify all preprocessing outputs (amendment 1.2)')

## Step 7 · Summary. Every row should show protocol 1.1 and `pilot? False`, and the verification table `yes` everywhere. Then tell the project thread you are done.

In [ ]:
import json, glob
import pandas as pd
from IPython.display import Markdown, display
rows = []
for f in sorted(glob.glob('results/data_audit/*/report.json')):
    r = json.load(open(f))
    c = r.get('counts', {})
    rows.append({'dataset': r['dataset'], 'rows kept': c.get('rows_out', r.get('host_hours')),
                 'protocol': r.get('protocol_version'), 'code': (r['run']['git_commit'] or '')[:7],
                 'built': r['run']['utc'], 'pilot?': bool(r.get('max_rows'))})
display(pd.DataFrame(rows))
near = []
for f in sorted(glob.glob('results/data_audit/*/near_duplicates.json')):
    s = json.load(open(f))
    for sig in ('sig3', 'sig2'):
        t = s['subsets'][sig].get('test', {})
        near.append({'dataset': s['dataset'], 'rounding': sig, 'test rows': t.get('rows'),
                     'unseen test rows (S7)': t.get('unseen_rows'), 'unseen share': round(t.get('unseen_share') or 0, 4),
                     'estimable': t.get('estimable')})
x = json.load(open('results/data_audit/cross_dataset_overlap.json'))
for d in ('csecicids2018_improved', 'unsw_nb15'):
    for sig in ('sig3', 'sig2'):
        t = x[d]['near_duplicates'][sig]
        near.append({'dataset': d + ' (vs CICIDS2017 train)', 'rounding': sig, 'test rows': t['rows'],
                     'unseen test rows (S7)': t['unseen_rows'], 'unseen share': round(t['unseen_share'], 4),
                     'estimable': t['estimable']})
display(pd.DataFrame(near))
display(Markdown(open('results/data_audit/verification.md').read()))